# 03 - Article Embeddings & Similar Articles (simple version)

Steps:
1. Load articles and transactions
2. Only use transactions BEFORE the last week (so we never peek at the discount week)
3. Turn article text into vectors (text embeddings)
4. Turn customer purchases into vectors (SVD)
5. For every article, find the 30 most similar articles
6. Save the tables

| Output table | Content |
|---|---|
| `hnm_article_text_embeddings` | article_id, text_embedding (used by 04 and 05 for matching) |
| `hnm_article_text_nn_candidates` | top-k similar articles for each article (used by 04) |
| `hnm_embedding_run_meta` | one row per run with counts and settings (used by 06 for the freshness check) |

In [0]:
%pip install -q torch
%pip install -q sentence-transformers

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import os
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from pyspark.sql import functions as F
from datetime import datetime

run_time = datetime.now()

In [0]:
# Settings
dbutils.widgets.text("transactions_table", "workspace.default.hnm_transactions_bronze")
dbutils.widgets.text("articles_table", "workspace.default.hnm_articles_bronze")
dbutils.widgets.text("output_schema", "workspace.hnm_discount")
dbutils.widgets.text("top_k_neighbors", "30")
dbutils.widgets.text("svd_components", "64")
dbutils.widgets.text("text_weight", "0.5")

transactions_table = dbutils.widgets.get("transactions_table")
articles_table = dbutils.widgets.get("articles_table")
output_schema = dbutils.widgets.get("output_schema")
top_k = int(dbutils.widgets.get("top_k_neighbors"))
n_svd_components = int(dbutils.widgets.get("svd_components"))
text_weight = float(dbutils.widgets.get("text_weight"))

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {output_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {output_schema}.embedding_cache")

text_embeddings_table = f"{output_schema}.hnm_article_text_embeddings"
nn_candidates_table = f"{output_schema}.hnm_article_text_nn_candidates"
run_meta_table = f"{output_schema}.hnm_embedding_run_meta"

def save_table(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)

In [0]:
# Load data
articles = spark.table(articles_table)
trans = spark.table(transactions_table)

# Cutoff = the Monday of the latest week. Transactions on/after this day are the discount week, so we skip them.
latest_day = pd.Timestamp(trans.agg(F.max("t_dat")).first()[0])
cutoff = latest_day.normalize() - pd.Timedelta(days=latest_day.weekday())
cutoff_str = cutoff.strftime("%Y-%m-%d")
print("latest transaction:", latest_day.date(), "| cutoff:", cutoff_str)

trans_pre = trans.filter(F.col("t_dat") < F.lit(cutoff_str).cast("date"))

latest transaction: 2020-09-06 | cutoff: 2020-08-31


In [0]:
# Build one sentence of text for every article
text_columns = ["prod_name", "product_type_name", "product_group_name", "graphical_appearance_name",
                "colour_group_name", "index_name", "section_name", "garment_group_name"]

articles_pd = articles.select(["article_id"] + text_columns).toPandas()
articles_pd["article_id"] = articles_pd["article_id"].astype(str)
articles_pd = articles_pd.sort_values("article_id").reset_index(drop=True)   # keep this sort, row order matters

articles_pd["text"] = articles_pd[text_columns].fillna("").astype(str).agg(" ".join, axis=1)

article_ids = articles_pd["article_id"].tolist()
sentences = articles_pd["text"].tolist()
N = len(article_ids)
print("articles:", N)

articles: 105542


In [0]:
# Text embeddings. Saved to a file so we do not re-encode every run.
cache_file = f"/Volumes/{output_schema.replace('.', '/')}/embedding_cache/text_emb_tensor.pt"

text_emb = None
if os.path.exists(cache_file):
    text_emb = torch.load(cache_file, map_location="cpu")
    if text_emb.shape[0] != N:          # articles table changed, cache is out of date
        text_emb = None

if text_emb is None:
    model = SentenceTransformer("all-MiniLM-L6-v2")
    vectors = model.encode(sentences, batch_size=256, show_progress_bar=True, normalize_embeddings=True)
    text_emb = torch.tensor(vectors, dtype=torch.float32)
    torch.save(text_emb, cache_file)

print("text embeddings:", tuple(text_emb.shape))

text embeddings: (105542, 384)


In [0]:
# Behavior embeddings: which customers bought which article (before the cutoff)
pairs = trans_pre.groupBy("customer_id", "article_id").count().toPandas()
pairs["article_id"] = pairs["article_id"].astype(str)
pairs = pairs[pairs["article_id"].isin(set(article_ids))]

article_codes, behavior_article_ids = pd.factorize(pairs["article_id"], sort=True)
customer_codes, customers = pd.factorize(pairs["customer_id"], sort=True)

# rows = articles, columns = customers, value = how many times bought
M = csr_matrix((pairs["count"].to_numpy(dtype="float64"), (article_codes, customer_codes)),
               shape=(len(behavior_article_ids), len(customers)))

svd = TruncatedSVD(n_components=n_svd_components, random_state=42)
behavior_small = svd.fit_transform(M)
explained = float(svd.explained_variance_ratio_.sum())
print("matrix shape:", M.shape, "| explained variance:", round(explained, 3))

# Put the behavior vectors into the same article order as the text vectors.
# Articles that never sold before the cutoff stay as zeros.
position = {a: i for i, a in enumerate(article_ids)}
behavior_emb = np.zeros((N, n_svd_components), dtype=np.float32)
has_behavior = np.zeros(N, dtype=bool)
for a, vec in zip(behavior_article_ids, behavior_small):
    i = position[a]
    behavior_emb[i] = vec
    has_behavior[i] = True

# make every vector length 1 so dot product = cosine similarity
lengths = np.linalg.norm(behavior_emb, axis=1, keepdims=True)
lengths[lengths == 0] = 1.0
behavior_emb = behavior_emb / lengths

n_behavior = int(has_behavior.sum())
print("articles with behavior:", n_behavior, "of", N)

matrix shape: (101696, 1342770) | explained variance: 0.068
articles with behavior: 101696 of 105542


In [0]:
# Find the most similar articles for every article.
# Rules:
#  - Candidates must have behavior (they sold before the cutoff).
#  - If the article has behavior:  score = text_weight * text_sim + (1 - text_weight) * behavior_sim
#  - If the article has no behavior (new article): score = text_sim only
behavior_tensor = torch.tensor(behavior_emb)
has_behavior_tensor = torch.tensor(has_behavior)

# -inf for candidates without behavior so they can never be picked
candidate_penalty = torch.zeros(N)
candidate_penalty[~has_behavior_tensor] = float("-inf")

k = min(top_k, n_behavior - 1)
article_ids_array = np.array(article_ids)
batch_size = 1000
pieces = []

for start in range(0, N, batch_size):
    end = min(start + batch_size, N)
    n_rows = end - start

    text_sim = text_emb[start:end] @ text_emb.T
    behavior_sim = behavior_tensor[start:end] @ behavior_tensor.T

    # weight per article in this batch
    weight = torch.ones(n_rows, 1)
    weight[has_behavior_tensor[start:end]] = text_weight

    score = weight * text_sim + (1 - weight) * behavior_sim
    score = score + candidate_penalty          # drop candidates without behavior
    for i in range(n_rows):
        score[i, start + i] = float("-inf")    # an article is not its own neighbor

    top_scores, top_index = torch.topk(score, k=k, dim=1)     # already sorted best first
    top_text = torch.gather(text_sim, 1, top_index)
    top_behavior = torch.gather(behavior_sim, 1, top_index)

    anchor_has_behavior = np.repeat(has_behavior[start:end], k)

    pieces.append(pd.DataFrame({
        "anchor_id": np.repeat(article_ids_array[start:end], k),
        "candidate_id": article_ids_array[top_index.numpy().ravel()],
        "text_sim": top_text.numpy().ravel().astype("float64"),
        "behavior_sim": np.where(anchor_has_behavior, top_behavior.numpy().ravel(), np.nan),
        "final_score": top_scores.numpy().ravel().astype("float64"),
        "rank_overall": np.tile(np.arange(1, k + 1), n_rows),
        "score_basis": np.where(anchor_has_behavior, "blend", "text_only"),
    }))

neighbors = pd.concat(pieces, ignore_index=True)
print("neighbor rows:", len(neighbors), "| articles:", neighbors["anchor_id"].nunique())

neighbor rows: 3166260 | articles: 105542


In [0]:
# Quick checks
assert (neighbors["anchor_id"] != neighbors["candidate_id"]).all(), "an article is its own neighbor"
assert not neighbors.duplicated(["anchor_id", "candidate_id"]).any(), "duplicate pairs"
assert neighbors["candidate_id"].isin(set(behavior_article_ids)).all(), "a candidate has no behavior"
assert neighbors["anchor_id"].nunique() == N, "some articles have no neighbors"
print("checks passed")

checks passed


In [0]:
# Save tables
def nan_to_null(df):
    return df.withColumn("behavior_sim", F.when(F.isnan("behavior_sim"), None).otherwise(F.col("behavior_sim")))

text_df = spark.createDataFrame(
    [(a, v.tolist()) for a, v in zip(article_ids, text_emb.numpy())],
    "article_id string, text_embedding array<float>")
save_table(text_df, text_embeddings_table)

nn_df = spark.createDataFrame(neighbors).withColumn(
    "behavior_sim", F.when(F.isnan("behavior_sim"), None).otherwise(F.col("behavior_sim")))
save_table(nn_df, nn_candidates_table)

meta = pd.DataFrame([{
    "run_ts": run_time,
    "behavior_cutoff_date": cutoff_str,
    "n_articles": N,
    "n_articles_with_behavior": n_behavior,
    "top_k_effective": int(k),
    "text_weight": text_weight,
    "svd_components": n_svd_components,
    "svd_explained_variance": explained,
}])
save_table(spark.createDataFrame(meta), run_meta_table)

print("done")

done


In [0]:
print (meta)

                      run_ts  ... svd_explained_variance
0 2026-10-04 01:33:11.375668  ...               0.068385

[1 rows x 8 columns]
